# 03. 그래프 작성

논문 Figure용 그래프 생성. 실제 결과 CSV가 없으면 논문 기대값으로 데모 플롯을 생성합니다.

- Fig 1: 예산 민감도 곡선 (Budget vs Accuracy)
- Fig 2: Ablation - Score 성분 기여도
- Fig 3: 메모리-정확도 트레이드오프 산점도
- Fig 4: 태스크별 성능 비교 막대 그래프

In [ ]:
import sys, os, glob
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

RESULTS_DIR = 'results'
FIG_DIR     = 'results/figures'
os.makedirs(FIG_DIR, exist_ok=True)

# 방법별 고정 색상
METHOD_COLORS = {
    'FULLKV':    '#2d6a4f',
    'STREAMING': '#b5e48c',
    'H2O':       '#74c69d',
    'SNAPKV':    '#52b788',
    'PYRAMIDKV': '#40916c',
    'ADAKV':     '#1b4332',
    'OURS':      '#e63946',
}

def load_csv(prefix):
    files = sorted(glob.glob(f'{RESULTS_DIR}/{prefix}*.csv'))
    if not files:
        return pd.DataFrame()
    return pd.read_csv(files[-1])

print('그래프 환경 준비 완료')
print(f'저장 경로: {os.path.abspath(FIG_DIR)}')

## Fig 1: 예산 민감도 곡선

In [ ]:
df4 = load_csv('exp4_budget_sensitivity_qwen3-4b')

# 데이터 준비
if df4.empty or 'Budget' not in df4.columns:
    print('실제 결과 없음 - 논문 기대값으로 데모')
    budgets = [10, 15, 20, 25, 30, 40, 50]
    fullkv  = [37.5] * 7
    adakv   = [30.2, 32.8, 35.1, 36.3, 36.9, 37.3, 37.4]
    ours    = [32.1, 34.5, 36.6, 37.2, 37.5, 37.6, 37.6]
else:
    budgets = [int(b.replace('%','')) for b in df4['Budget']]
    fullkv  = df4['Full_KV'].tolist()
    adakv   = df4['ADAKV'].tolist()
    ours    = df4['OURS'].tolist()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(budgets, fullkv, '--',  color='gray',    label='Full KV (upper bound)', lw=1.5, alpha=0.7)
ax.plot(budgets, adakv,  '-o', color='#1b4332', label='Ada-KV',               lw=2,   ms=6)
ax.plot(budgets, ours,   '-s', color='#e63946', label='Ours (Hybrid)',         lw=2.5, ms=7)

# 30% 포인트 강조
ax.axvline(x=30, color='#e63946', ls=':', alpha=0.4)
ax.annotate('30%: Full KV\nequivalent',
            xy=(30, fullkv[0]), xytext=(33, fullkv[0]-2.5),
            color='#e63946', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#e63946', lw=1.2))

ax.set_xlabel('KV Cache Budget (%)')
ax.set_ylabel('Average Score (F1 / ROUGE-L)')
ax.set_title('Budget Sensitivity (Qwen3-4B, LongBench)')
ax.legend(loc='lower right')
ax.set_xlim(8, 52)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig1_budget_sensitivity.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig1_budget_sensitivity.png', bbox_inches='tight')
plt.show()
print(f'Saved: fig1_budget_sensitivity')

## Fig 2: Ablation - Score 성분 기여도

In [ ]:
df2 = load_csv('exp2_score_ablation_qwen3-4b')

if df2.empty or 'Avg' not in df2.columns:
    print('데모 데이터 사용')
    conditions = ['Full (All 4)', 'w/o Attention', 'w/o Semantic', 'w/o Entropy', 'w/o Position']
    scores     = [36.6, 31.2, 34.1, 34.8, 35.3]
else:
    conditions = df2['Condition'].tolist()
    scores     = df2['Avg'].tolist()

colors = ['#e63946' if i == 0 else '#74c69d' for i in range(len(conditions))]

fig, ax = plt.subplots(figsize=(8.5, 4.5))
bars = ax.bar(range(len(conditions)), scores, color=colors, edgecolor='white', width=0.6)
ax.axhline(y=scores[0], color='#e63946', ls='--', alpha=0.4, lw=1.2)

for i, (bar, score) in enumerate(zip(bars, scores)):
    # 점수 표시
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{score:.1f}', ha='center', va='bottom', fontsize=10)
    # delta 표시 (Full 제외)
    if i > 0:
        delta = score - scores[0]
        ax.text(bar.get_x() + bar.get_width()/2, score/2,
                f'{delta:.1f}', ha='center', va='center',
                color='white', fontsize=9, fontweight='bold')

# x축 레이블 줄임
xlabels = [c.replace('Full (All 4 signals)', 'Full\n(All 4)').replace('w/o ', 'w/o\n') for c in conditions]
ax.set_xticks(range(len(conditions)))
ax.set_xticklabels(xlabels, fontsize=10)
ax.set_ylabel('Average Score')
ax.set_title('Ablation: Hybrid Score Components (Qwen3-4B, Budget=20%)')
ax.set_ylim(min(scores) - 2, max(scores) + 2)

legend_elem = [
    mpatches.Patch(facecolor='#e63946', label='Full model (baseline)'),
    mpatches.Patch(facecolor='#74c69d', label='Component removed'),
]
ax.legend(handles=legend_elem, loc='lower right')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig2_ablation_score.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig2_ablation_score.png', bbox_inches='tight')
plt.show()
print('Saved: fig2_ablation_score')

## Fig 3: 메모리-정확도 트레이드오프 산점도

In [ ]:
df1 = load_csv('exp1_qwen3-4b_full')

if df1.empty or 'Avg' not in df1.columns:
    print('데모 데이터 사용')
    methods    = ['FullKV', 'StreamingLLM', 'H2O', 'SnapKV', 'PyramidKV', 'AdaKV', 'Ours']
    mem_red    = [0.0, 68.0, 80.0, 80.0, 80.0, 80.0, 73.0]
    avg_scores = [37.5, 26.2, 31.2, 33.8, 34.5, 35.1, 36.6]
else:
    methods    = df1['Method'].tolist()
    mem_red    = df1['Mem_Reduction_%'].tolist()
    avg_scores = df1['Avg'].tolist()

fig, ax = plt.subplots(figsize=(7.5, 5.5))

for m, mem, score in zip(methods, mem_red, avg_scores):
    color  = METHOD_COLORS.get(m.upper(), '#aaa')
    is_ours = m.upper() == 'OURS'
    size   = 220 if is_ours else 110
    marker = '*'  if is_ours else 'o'
    zorder = 6    if is_ours else 5
    ax.scatter(mem, score, c=color, s=size, marker=marker,
               zorder=zorder, edgecolors='white', linewidth=0.8)
    y_off = 0.25 if is_ours else -0.45
    ax.annotate(m, (mem, score), xytext=(mem + 1.0, score + y_off), fontsize=9.5)

ax.set_xlabel('Memory Reduction (%)')
ax.set_ylabel('Average Score (F1 / ROUGE-L)')
ax.set_title('Memory-Accuracy Trade-off (Qwen3-4B, Budget=20%)')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig3_tradeoff.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig3_tradeoff.png', bbox_inches='tight')
plt.show()
print('Saved: fig3_tradeoff')

## Fig 4: 태스크별 성능 비교

In [ ]:
TASK_KEYS   = ['narrativeqa', 'qasper', 'multifieldqa_en', 'hotpotqa', '2wikimqa', 'gov_report', 'qmsum']
TASK_LABELS = ['NarrQA', 'Qasper', 'MultiQA', 'HotpotQA', '2WikiQA', 'GovRep', 'QMSum']

df1 = load_csv('exp1_qwen3-4b_full')

# 표시할 방법 3개만 선택
SHOW_METHODS = ['FULLKV', 'ADAKV', 'OURS']

if df1.empty or 'Method' not in df1.columns:
    print('데모 데이터 사용')
    plot_data = {
        'FULLKV': [24.3, 42.1, 45.2, 38.7, 35.9, 29.8, 46.5],
        'ADAKV':  [22.8, 40.3, 43.1, 36.5, 34.1, 28.4, 40.9],
        'OURS':   [23.7, 41.2, 44.5, 37.9, 35.2, 29.3, 44.4],
    }
else:
    plot_data = {}
    for _, row in df1.iterrows():
        m = str(row['Method'])
        if m in SHOW_METHODS:
            plot_data[m] = [float(row.get(t, 0)) for t in TASK_KEYS]

x       = np.arange(len(TASK_LABELS))
n_meth  = len(plot_data)
width   = 0.22
offsets = np.linspace(-(n_meth-1)*width/2, (n_meth-1)*width/2, n_meth)

fig, ax = plt.subplots(figsize=(11, 5))
for i, (method, scores) in enumerate(plot_data.items()):
    color   = METHOD_COLORS.get(method, '#aaa')
    is_ours = method == 'OURS'
    ec      = '#e63946' if is_ours else 'white'
    lw      = 1.5 if is_ours else 0
    ax.bar(x + offsets[i], scores, width, label=method,
           color=color, edgecolor=ec, linewidth=lw, alpha=0.88)

ax.set_xticks(x)
ax.set_xticklabels(TASK_LABELS, rotation=10)
ax.set_ylabel('Score (F1 / ROUGE-L)')
ax.set_title('Per-Task Performance (Qwen3-4B, Budget=20%)')
ax.legend(loc='upper right', ncol=n_meth)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_per_task.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig4_per_task.png', bbox_inches='tight')
plt.show()
print('Saved: fig4_per_task')

## Fig 5: 레이어 할당 전략 비교 (실험 3)

In [ ]:
df3 = load_csv('exp3_allocation_qwen3-4b')

if df3.empty or 'Avg_Score' not in df3.columns:
    print('데모 데이터 사용')
    strategies = ['Entropy-Driven (Ours)', 'Uniform (Equal)', 'Static Pyramidal', 'Random']
    scores_3   = [36.6, 34.3, 34.9, 32.8]
else:
    strategies = df3['Strategy'].tolist()
    scores_3   = df3['Avg_Score'].tolist()

colors_3 = ['#e63946'] + ['#74c69d'] * (len(strategies) - 1)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(range(len(strategies)), scores_3, color=colors_3, edgecolor='white', width=0.55)
ax.axhline(y=scores_3[0], color='#e63946', ls='--', alpha=0.4, lw=1.2)

for i, (bar, score) in enumerate(zip(bars, scores_3)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{score:.1f}', ha='center', va='bottom', fontsize=10)
    if i > 0:
        delta = score - scores_3[0]
        ax.text(bar.get_x() + bar.get_width()/2, score/2,
                f'{delta:.1f}', ha='center', va='center',
                color='white', fontsize=9, fontweight='bold')

short_labels = [s.split('(')[0].strip() for s in strategies]
ax.set_xticks(range(len(strategies)))
ax.set_xticklabels(short_labels, fontsize=10)
ax.set_ylabel('Average Score')
ax.set_title('Layer Budget Allocation Strategies (Qwen3-4B, Budget=20%)')
ax.set_ylim(min(scores_3) - 2, max(scores_3) + 2)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig5_allocation.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig5_allocation.png', bbox_inches='tight')
plt.show()
print('Saved: fig5_allocation')

In [ ]:
# 저장된 파일 목록
import os
print(f'저장된 그래프 ({FIG_DIR}/):')
for f in sorted(os.listdir(FIG_DIR)):
    size = os.path.getsize(f'{FIG_DIR}/{f}') / 1024
    print(f'  {f:<40} {size:6.1f} KB')